# PPE Detection Project - Google Colab

Notebook này được thiết kế để chạy **thuần trên Google Colab**: clone project từ GitHub nhánh `main`, cài dependency, chuẩn bị dataset Kaggle/Roboflow, train YOLO, evaluate và detect ảnh mẫu.

**Trước khi chạy:** vào `Runtime → Change runtime type → GPU` nếu muốn train nhanh hơn.

## 1. Clone project từ GitHub nhánh main

Điền `GIT_REPO_URL` bằng link GitHub repo bạn đang làm. Notebook sẽ clone trực tiếp nhánh `main`, tự tìm thư mục `ppe_detection_project`, rồi chạy toàn bộ các bước còn lại trên Colab.

> Notebook này không dùng luồng upload zip/local nữa để giữ workflow thuần Colab + GitHub.

In [ ]:
from pathlib import Path
import os

# TODO: Dán link GitHub repo bạn đang làm vào đây để Colab clone trực tiếp từ branch main.
# Ví dụ: GIT_REPO_URL = "https://github.com/<username>/computer-vision.git"
GIT_REPO_URL = "https://github.com/<username>/computer-vision.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

if not GIT_REPO_URL or "<username>" in GIT_REPO_URL:
    raise ValueError("Hãy thay GIT_REPO_URL bằng link GitHub repo thật trước khi chạy notebook.")
if BRANCH != "main":
    print(f"Warning: BRANCH đang là {BRANCH!r}. Theo yêu cầu hiện tại nên chạy branch 'main'.")

CLONE_DIR = Path("/content/computer-vision")
!rm -rf {CLONE_DIR}
!git clone --branch {BRANCH} {GIT_REPO_URL} {CLONE_DIR}

candidates = [CLONE_DIR / "ppe_detection_project", *CLONE_DIR.rglob("ppe_detection_project")]
candidates = [path for path in candidates if path.exists()]
if not candidates:
    raise FileNotFoundError("Không tìm thấy thư mục ppe_detection_project trong repo vừa clone.")

PROJECT_DIR = candidates[0]
os.chdir(PROJECT_DIR)
print("Project dir:", PROJECT_DIR)
!pwd
!git -C {CLONE_DIR} status --short --branch
!find . -maxdepth 2 -type f | sort | head -50

## 2. Cài đặt dependencies

In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Cấu hình Kaggle và tải dataset

Dataset dùng trong project: `snehilsanyal/construction-site-safety-image-dataset-roboflow`.

Nếu chưa có credential Kaggle:
1. Vào Kaggle → Account → Create New API Token để tải `kaggle.json`.
2. Chạy cell dưới và upload file `kaggle.json`.

In [ ]:
from pathlib import Path
from google.colab import files
import os

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"

if not kaggle_json.exists():
    print("Upload kaggle.json...")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("Bạn cần upload đúng file kaggle.json")
    (Path("kaggle.json")).replace(kaggle_json)

os.chmod(kaggle_json, 0o600)
print("Kaggle credential ready:", kaggle_json)

In [ ]:
import subprocess

DATASET_DIR = Path("/content/datasets/construction_site_safety")
DATA_YAML = PROJECT_DIR / "data" / "data.yaml"

subprocess.run([
    "python", str(PROJECT_DIR / "src" / "prepare_css_dataset.py"),
    "--download",
    "--output", str(DATASET_DIR),
    "--yaml-output", str(DATA_YAML),
    "--copy",
    "--force",
], check=True)

print(DATA_YAML.read_text(encoding="utf-8"))

## 4. Train YOLO

Có thể đổi `MODEL`, `EPOCHS`, `IMGSZ`, `BATCH`. Colab GPU thường dùng `DEVICE = 0`; nếu không có GPU thì đặt `DEVICE = "cpu"`.

In [ ]:
import subprocess

MODEL = "yolov8n.pt"   # hoặc "yolo11n.pt"
EPOCHS = 50
IMGSZ = 640
BATCH = 16
DEVICE = 0 if torch.cuda.is_available() else "cpu"

DATA_YAML = PROJECT_DIR / "data" / "data.yaml"
TRAIN_PROJECT = PROJECT_DIR / "runs" / "train"
TRAIN_NAME = "ppe_yolo"

subprocess.run([
    "python", str(PROJECT_DIR / "src" / "train.py"),
    "--data", str(DATA_YAML),
    "--model", MODEL,
    "--epochs", str(EPOCHS),
    "--imgsz", str(IMGSZ),
    "--batch", str(BATCH),
    "--device", str(DEVICE),
    "--project", str(TRAIN_PROJECT),
    "--name", TRAIN_NAME,
], check=True)

best_candidates = sorted(TRAIN_PROJECT.glob("*/weights/best.pt"), key=lambda path: path.stat().st_mtime, reverse=True)
if not best_candidates:
    raise FileNotFoundError(f"Không tìm thấy best.pt trong {TRAIN_PROJECT}. Hãy kiểm tra cell train phía trên.")
BEST_MODEL = best_candidates[0]
print("BEST_MODEL =", BEST_MODEL)

## 5. Evaluate model

In [ ]:
import subprocess
from pathlib import Path

DATA_YAML = PROJECT_DIR / "data" / "data.yaml"
TRAIN_PROJECT = PROJECT_DIR / "runs" / "train"
if "BEST_MODEL" not in globals() or not Path(BEST_MODEL).is_file():
    best_candidates = sorted(TRAIN_PROJECT.glob("*/weights/best.pt"), key=lambda path: path.stat().st_mtime, reverse=True)
    if not best_candidates:
        raise FileNotFoundError(f"Không tìm thấy best.pt trong {TRAIN_PROJECT}. Hãy chạy lại cell Train YOLO trước.")
    BEST_MODEL = best_candidates[0]

EVAL_PROJECT = PROJECT_DIR / "runs" / "evaluate"
print("Evaluating model:", BEST_MODEL)

subprocess.run([
    "python", str(PROJECT_DIR / "src" / "evaluate.py"),
    "--model", str(BEST_MODEL),
    "--data", str(DATA_YAML),
    "--split", "val",
    "--device", str(DEVICE),
    "--project", str(EVAL_PROJECT),
], check=True)

## 6. Detect ảnh upload trong Colab

In [ ]:
import subprocess
from google.colab import files
from IPython.display import Image as IPImage, display
from pathlib import Path

if "BEST_MODEL" not in globals() or not Path(BEST_MODEL).is_file():
    TRAIN_PROJECT = PROJECT_DIR / "runs" / "train"
    best_candidates = sorted(TRAIN_PROJECT.glob("*/weights/best.pt"), key=lambda path: path.stat().st_mtime, reverse=True)
    if not best_candidates:
        raise FileNotFoundError(f"Không tìm thấy best.pt trong {TRAIN_PROJECT}. Hãy chạy cell Train YOLO trước.")
    BEST_MODEL = best_candidates[0]

uploaded = files.upload()
if not uploaded:
    raise FileNotFoundError("Chưa upload ảnh.")

image_path = Path(next(iter(uploaded.keys()))).resolve()
OUTPUT_DIR = PROJECT_DIR / "runs" / "detect" / "colab_image"

subprocess.run([
    "python", str(PROJECT_DIR / "src" / "detect_image.py"),
    "--model", str(BEST_MODEL),
    "--source", str(image_path),
    "--output", str(OUTPUT_DIR),
    "--conf", "0.25",
], check=True)

output_path = OUTPUT_DIR / f"{image_path.stem}_detected{image_path.suffix}"
display(IPImage(filename=str(output_path)))

## 7. Lưu kết quả

Có thể tải trực tiếp file zip hoặc copy sang Google Drive.

In [ ]:
!zip -r /content/ppe_runs.zip {PROJECT_DIR / 'runs'} {PROJECT_DIR / 'data' / 'data.yaml'}
files.download('/content/ppe_runs.zip')